In [1]:
"""
prepare_gdfas_offline_bundle.py
================================
Run this in an INTERNET-ENABLED Kaggle notebook (CPU is fine). It produces

    /kaggle/working/gdfas_offline_bundle.zip

containing everything the offline, no-internet GPU notebook needs to run
GD-FAS (gdfas_experiments.py):

    gdfas_offline_bundle/
    |-- wheels/            pip wheels for ftfy + regex (and their deps) --
    |                      the only GD-FAS requirements NOT preinstalled in
    |                      Kaggle's image (torch/torchvision/sklearn/scipy/
    |                      numpy/PIL are already there; do NOT install the
    |                      repo's pinned old versions, they'd fight Kaggle's)
    |-- clip/ViT-B-16.pt   OpenAI CLIP checkpoint. GD-FAS's clip.load()
    |                      checks ~/.cache/clip first and verifies sha256,
    |                      so pre-seeding it means zero network calls.
    |-- GD-FAS/            the repo itself (cloned here so the offline
                           notebook doesn't need a separate upload)

Then: Kaggle sidebar -> "Add Data" -> upload gdfas_offline_bundle.zip as a
private dataset, and attach it to the offline notebook.
"""
# %% CELL 0 -- build the bundle
import os
import shutil
import subprocess
import sys
import urllib.request

BUNDLE_DIR = "/kaggle/working/gdfas_offline_bundle"
WHEELS_DIR = os.path.join(BUNDLE_DIR, "wheels")
CLIP_DIR = os.path.join(BUNDLE_DIR, "clip")
REPO_DIR = os.path.join(BUNDLE_DIR, "GD-FAS")

# EDIT if the repo moves; this is the official ICCV 2025 implementation.
GDFAS_GIT_URL = "https://github.com/SeungjinJung/GD-FAS.git"

# sha256 prefix in the URL is what clip.load() verifies against -- keep as is.
CLIP_VITB16_URL = ("https://openaipublic.azureedge.net/clip/models/"
                   "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f/ViT-B-16.pt")

os.makedirs(WHEELS_DIR, exist_ok=True)
os.makedirs(CLIP_DIR, exist_ok=True)

# 1) wheels -- `pip download` pulls each package AND its dependencies as
# wheels, so the offline `pip install --no-index` resolves fully local.
subprocess.check_call([
    sys.executable, "-m", "pip", "download",
    "ftfy", "regex",
    "-d", WHEELS_DIR,
])
print("wheels:", sorted(os.listdir(WHEELS_DIR)))

# 2) CLIP ViT-B/16 checkpoint (~340MB)
ckpt_path = os.path.join(CLIP_DIR, "ViT-B-16.pt")
if not os.path.isfile(ckpt_path):
    print("downloading ViT-B-16.pt ...")
    urllib.request.urlretrieve(CLIP_VITB16_URL, ckpt_path)
print("clip checkpoint:", ckpt_path, os.path.getsize(ckpt_path), "bytes")

# 3) the GD-FAS repo (drop .git, it's dead weight offline)
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth", "1", GDFAS_GIT_URL, REPO_DIR])
    shutil.rmtree(os.path.join(REPO_DIR, ".git"), ignore_errors=True)
print("repo files:", sorted(os.listdir(REPO_DIR)))

# 4) zip it
zip_path = shutil.make_archive("/kaggle/working/gdfas_offline_bundle", "zip",
                               root_dir="/kaggle/working", base_dir="gdfas_offline_bundle")
print("\nDONE ->", zip_path)
print("Upload this zip as a private Kaggle dataset and attach it to the offline notebook.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.7/331.7 kB 22.9 MB/s eta 0:00:00
Saved ./gdfas_offline_bundle/wheels/ftfy-6.3.1-py3-none-any.whl
Saved ./gdfas_offline_bundle/wheels/regex-2026.9.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
Saved ./gdfas_offline_bundle/wheels/wcwidth-0.8.3-py3-none-any.whl
Successfully downloaded ftfy regex wcwidth
wheels: ['ftfy-6.3.1-py3-none-any.whl', 'regex-2026.9.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl', 'wcwidth-0.8.3-py3-none-any.whl']
downloading ViT-B-16.pt ...
clip checkpoint: /kaggle/working/gdfas_offline_bundle/clip/ViT-B-16.pt 350837078 bytes


Cloning into '/kaggle/working/gdfas_offline_bundle/GD-FAS'...


repo files: ['GD-FAS.py', 'LICENSE', 'README.md', 'data', 'models', 'requirements.txt', 'run.sh', 'utils']

DONE -> /kaggle/working/gdfas_offline_bundle.zip
Upload this zip as a private Kaggle dataset and attach it to the offline notebook.
